In [16]:
import pandas as pd
import numpy as np
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [27]:
data = pd.read_csv('Ecommerce_FAQs.csv')
data.head()

,prompt,response
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...


In [28]:
#preprocess text with nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Function to tokenize and preprocess text
def preprocess_text(text):
    tokens = word_tokenize(text.lower())  # Tokenize and convert to lowercase
    # Remove stopwords and lemmatize
    filtered_tokens = [lemmatizer.lemmatize(word) for word in tokens if word.isalpha() and word not in stop_words]
    return filtered_tokens

# Apply preprocessing to 'prompt' and 'response' columns
data['tokenized_prompt'] = data['prompt'].apply(preprocess_text)
data['tokenized_response'] = data['response'].apply(preprocess_text)

print("Tokenized prompts:")
print(data['tokenized_prompt'].head())
print("\nTokenized responses:")
print(data['tokenized_response'].head())

#initialize chatbot
#from nltk.chat.util import Chat, reflections

Tokenized prompts:
0            [create, account]
1    [payment, method, accept]
2               [track, order]
3             [return, policy]
4              [cancel, order]
Name: tokenized_prompt, dtype: object

Tokenized responses:
0    [create, account, click, button, top, right, c...
1    [accept, major, credit, card, debit, card, pay...
2    [track, order, logging, account, navigating, h...
3    [return, policy, allows, return, product, with...
4    [cancel, order, shipped, yet, please, contact,...
Name: tokenized_response, dtype: object


In [29]:
# Cosine Similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Join tokenized words back into strings for TF-IDF vectorizer
data['processed_prompt'] = data['tokenized_prompt'].apply(lambda x: ' '.join(x))
data['processed_response'] = data['tokenized_response'].apply(lambda x: ' '.join(x))

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the processed prompts
prompt_vectors = tfidf_vectorizer.fit_transform(data['processed_prompt'])

# Transform the processed responses using the *same* fitted vectorizer
response_vectors = tfidf_vectorizer.transform(data['processed_response'])

# Calculate cosine similarity for each prompt-response pair
# We'll iterate through each pair and calculate similarity. This is more straightforward than a full matrix for pair-wise.
cosine_similarities = []
for i in range(len(data)):
    similarity = cosine_similarity(prompt_vectors[i], response_vectors[i])[0][0]
    cosine_similarities.append(similarity)

data['cosine_similarity'] = cosine_similarities

print("Data with Cosine Similarities:")
display(data[['prompt', 'response', 'cosine_similarity']].head())

Data with Cosine Similarities:


,prompt,response,cosine_similarity
0,How can I create an account?,"To create an account, click on the 'Sign Up' b...",0.821023
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and...",0.620863
2,How can I track my order?,You can track your order by logging into your ...,0.786619
3,What is your return policy?,Our return policy allows you to return product...,0.624777
4,Can I cancel my order?,You can cancel your order if it has not been s...,0.510801


In [30]:
# Function to get the most similar response for a given query
def get_chatbot_response(user_query):
    # Preprocess the user query using the same function
    processed_query_tokens = preprocess_text(user_query)
    processed_query = ' '.join(processed_query_tokens)

    if not processed_query:
        return "I'm sorry, I couldn't understand your query. Can you rephrase that?"

    # Transform the user query using the *fitted* TF-IDF vectorizer
    user_query_vector = tfidf_vectorizer.transform([processed_query])

    # Calculate cosine similarities between the user query and all existing prompt vectors
    similarities = cosine_similarity(user_query_vector, prompt_vectors)

    # Get the index of the most similar prompt
    most_similar_idx = similarities.argmax()

    # Retrieve the corresponding response
    return data['response'].iloc[most_similar_idx]

print("Chatbot response function defined!")

Chatbot response function defined!


In [31]:
# Simple Chatbot Interaction Loop
#print("Hello! I'm your FAQ Chatbot. Type 'exit' to end the conversation.")

#while True:
#    user_input = input("You: ")
#    if user_input.lower() == 'exit':
#        print("Chatbot: Goodbye!")
#        break

#    response = get_chatbot_response(user_input)
#    print(f"Chatbot: {response}")

Hello! I'm your FAQ Chatbot. Type 'exit' to end the conversation.
You: How to create account
Chatbot: To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.
You: Hot to delete account
Chatbot: To delete your account, head to your profile page, click on the settings button. You will find a section called "Account" where you can edit your account info, or delete it permanently, click on the "Delete Account" button
You: thank you
Chatbot: To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.
You: 432623
Chatbot: I'm sorry, I couldn't understand your query. Can you rephrase that?
You: exit
Chatbot: Goodbye!


In [33]:
from flask import Flask, request, render_template_string

# Define the HTML template as a string
html_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>FAQ Chatbot</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 0; padding: 0; background-color: #f4f4f4; display: flex; justify-content: center; align-items: center; min-height: 100vh; }
        .chat-container { background-color: #fff; border-radius: 8px; box-shadow: 0 2px 10px rgba(0, 0, 0, 0.1); width: 100%; max-width: 600px; display: flex; flex-direction: column; height: 70vh; overflow: hidden; }
        .chat-header { background-color: #007bff; color: white; padding: 15px; text-align: center; font-size: 1.2em; border-top-left-radius: 8px; border-top-right-radius: 8px; }
        .chat-box { flex-grow: 1; padding: 20px; overflow-y: auto; border-bottom: 1px solid #eee; scroll-behavior: smooth; }
        .message { margin-bottom: 15px; display: flex; }
        .message.user { justify-content: flex-end; }
        .message.chatbot { justify-content: flex-start; }
        .message .bubble { padding: 10px 15px; border-radius: 20px; max-width: 70%; word-wrap: break-word; }
        .message.user .bubble { background-color: #007bff; color: white; }
        .message.chatbot .bubble { background-color: #e2e6ea; color: #333; }
        .chat-input { display: flex; padding: 15px; border-top: 1px solid #eee; }
        .chat-input input { flex-grow: 1; padding: 10px; border: 1px solid #ddd; border-radius: 20px; margin-right: 10px; font-size: 1em; }
        .chat-input button { background-color: #28a745; color: white; border: none; padding: 10px 20px; border-radius: 20px; cursor: pointer; font-size: 1em; }
        .chat-input button:hover { background-color: #218838; }
    </style>
</head>
<body>
    <div class="chat-container">
        <div class="chat-header">FAQ Chatbot</div>
        <div class="chat-box" id="chat-box"></div>
        <div class="chat-input">
            <input type="text" id="user-input" placeholder="Type your message...">
            <button onclick="sendMessage()">Send</button>
        </div>
    </div>

    <script>
        function addMessage(sender, text) {
            const chatBox = document.getElementById('chat-box');
            const messageDiv = document.createElement('div');
            messageDiv.classList.add('message', sender);
            const bubbleDiv = document.createElement('div');
            bubbleDiv.classList.add('bubble');
            bubbleDiv.textContent = text;
            messageDiv.appendChild(bubbleDiv);
            chatBox.appendChild(messageDiv);
            chatBox.scrollTop = chatBox.scrollHeight; // Scroll to bottom
        }

        async function sendMessage() {
            const userInput = document.getElementById('user-input');
            const userMessage = userInput.value.trim();

            if (userMessage) {
                addMessage('user', userMessage);
                userInput.value = '';

                const response = await fetch('/chat', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json'
                    },
                    body: JSON.stringify({ message: userMessage })
                });
                const data = await response.json();
                addMessage('chatbot', data.response);
            }
        }

        // Initial welcome message
        addMessage('chatbot', "Hello! I'm your FAQ Chatbot. Ask me anything about our e-commerce site!");
    </script>
</body>
</html>
"""

# Flask requires templates to be in a 'templates' directory
# We'll create a simple folder structure for this.

# Create the 'templates' directory if it doesn't exist
if not os.path.exists('templates'):
    os.makedirs('templates')

# Save the HTML template to a file inside the 'templates' directory
with open('templates/index.html', 'w') as f:
    f.write(html_template)

print("HTML template saved to 'templates/index.html'")

HTML template saved to 'templates/index.html'


#### Flask Application Code

This code sets up our Flask application:

*   An `@app.route('/')` decorator defines the home page, which will render our `index.html` template.
*   An `@app.route('/chat', methods=['POST'])` decorator handles incoming user messages, processes them using our `get_chatbot_response` function, and returns the chatbot's reply as JSON.

In [37]:
app = Flask(__name__)


@app.route('/')
def index():
    # Render the HTML template (saved in 'templates/index.html')
    with open('templates/index.html', 'r') as f:
        return render_template_string(f.read())


@app.route('/chat', methods=['POST'])
def chat():
    user_message = request.json['message']
    chatbot_response = get_chatbot_response(user_message)
    return {'response': chatbot_response}

# run on localhost 5000
if __name__ == '__main__':
    app.run()




 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
